# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hammadkhaliq-del/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb

import duckdb
import pandas as pd
import numpy as np
import os
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('AccessToken')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # same mid-panel month as the ML-04 contract -- iterate here, never the sealed final month

DAILY_MONTH = f"{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print('Ready. Querying month:', MONTH)

Ready. Querying month: 2026-03


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page is worth reviewing first if it still pulls real search demand, it's been sitting untouched for a long time, and its ranking position is weak enough that a refresh could plausibly move it. Score = demand × staleness × position-weakness, all built from fields that were already true at the moment of scoring — no future window, no label-derived input.

**Reason codes it can output:**
- `stale_high_demand` — old content still pulling meaningful impressions
- `stale_weak_position` — old content ranking poorly (page 2+)
- `stale_high_demand_weak_position` — both conditions at once (highest priority)
- `fresh_low_priority` — recently updated or low demand, score near zero

Before coding the score, I check the two signals it leans on against a real outcome (position) so the rule isn't just an assumption dressed up as logic.

### Signal check 1 — staleness (behind FlyRank's real refresh flag)

**Signal:** `days_since_last_optimized`, bucketed. **Tested against:** `gsc_avg_position` (lower = better). If staleness is a real signal, stale pages should show worse average position than fresh ones.

In [2]:
staleness_check = con.sql(f"""
    WITH monthly AS (
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{DAILY_MONTH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    ),
    joined AS (
        SELECT m.avg_position_month,
               DATE '2026-03-31' - c.last_optimized_date AS days_since_update
        FROM monthly m
        LEFT JOIN read_parquet('{DIM_CONTENT}') c
          ON m.content_hash_id = c.content_hash_id AND m.client_hash_id = c.client_hash_id
        WHERE c.last_optimized_date IS NOT NULL
    )
    SELECT
        CASE
            WHEN days_since_update < 90 THEN '1_fresh_lt90d'
            WHEN days_since_update < 180 THEN '2_moderate_90_180d'
            WHEN days_since_update < 365 THEN '3_stale_180_365d'
            ELSE '4_very_stale_365d_plus'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(avg_position_month), 2) AS avg_position
    FROM joined
    GROUP BY 1
    ORDER BY 1
""").df()

print(staleness_check.to_string(index=False))
print()
print('Verdict: if avg_position rises (gets worse) from bucket 1 to bucket 4, staleness is CONFIRMED as a real signal.')
print('If it is flat or reverses, that is MIXED or OPPOSITE -- read the numbers above before deciding, do not assume.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

staleness_bucket     n  avg_position
   1_fresh_lt90d 39764         11.25

Verdict: if avg_position rises (gets worse) from bucket 1 to bucket 4, staleness is CONFIRMED as a real signal.
If it is flat or reverses, that is MIXED or OPPOSITE -- read the numbers above before deciding, do not assume.


**Verdict — staleness:** *(fill in after running: CONFIRMED / OPPOSITE / MIXED / FALSE, one word, then one sentence citing the actual bucket numbers above — e.g. "CONFIRMED: avg_position worsens from X in bucket 1 to Y in bucket 4, with n=... per bucket, so staleness tracks with worse ranking as the refresh-flag logic assumes.")*

### Signal check 2 — demand volume (behind FlyRank's quick-win flag)

**Signal:** `gsc_impressions` (monthly sum), bucketed. **Tested against:** `gsc_avg_position`. Quick-win logic assumes high-demand pages are worth prioritizing over low-demand ones regardless of position — this checks whether demand and position move together or independently.

In [3]:
demand_check = con.sql(f"""
    WITH monthly AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_month,
               AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{DAILY_MONTH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        CASE
            WHEN impressions_month < 50 THEN '1_low_lt50'
            WHEN impressions_month < 300 THEN '2_moderate_50_300'
            WHEN impressions_month < 1000 THEN '3_high_300_1000'
            ELSE '4_very_high_1000_plus'
        END AS demand_bucket,
        COUNT(*) AS n,
        ROUND(AVG(avg_position_month), 2) AS avg_position
    FROM monthly
    GROUP BY 1
    ORDER BY 1
""").df()

print(demand_check.to_string(index=False))
print()
print('Verdict: if avg_position IMPROVES (gets lower) as demand rises, that is expected/CONFIRMED --')
print('high-demand pages tend to already rank better, meaning demand alone finds already-good pages,')
print('not necessarily quick-win opportunities. Read the actual numbers before deciding the verdict word.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        demand_bucket     n  avg_position
           1_low_lt50 60624         16.96
    2_moderate_50_300 41448         21.18
      3_high_300_1000 29608         14.36
4_very_high_1000_plus 45058         11.02

Verdict: if avg_position IMPROVES (gets lower) as demand rises, that is expected/CONFIRMED --
high-demand pages tend to already rank better, meaning demand alone finds already-good pages,
not necessarily quick-win opportunities. Read the actual numbers before deciding the verdict word.


**Verdict — demand volume:** *(fill in after running: CONFIRMED / OPPOSITE / MIXED / FALSE, one word, then one sentence citing the bucket numbers — e.g. "MIXED: high-demand pages average better position, so demand alone doesn't isolate quick-wins; it needs to be paired with position-weakness in the rule, which is why the score below multiplies demand by a position condition rather than using demand alone.")*

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [4]:
scored = con.sql(f"""
    WITH monthly AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_month,
               SUM(gsc_clicks) AS clicks_month,
               AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{DAILY_MONTH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        m.content_hash_id,
        m.client_hash_id,
        m.impressions_month,
        m.clicks_month,
        m.avg_position_month,
        c.word_count,
        c.content_type,
        DATE '2026-03-31' - c.last_optimized_date AS days_since_update
    FROM monthly m
    LEFT JOIN read_parquet('{DIM_CONTENT}') c
      ON m.content_hash_id = c.content_hash_id AND m.client_hash_id = c.client_hash_id
    WHERE c.last_optimized_date IS NOT NULL
""").df()

print('Rows before scoring:', len(scored))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows before scoring: 39764


In [5]:
# --- The rule, coded as a transparent score. No fitted weights, readable on purpose. ---

# Conditions (all knowable at the decision moment -- no future window, no label-derived input)
is_stale = (scored['days_since_update'] >= 180).astype(int)
has_demand = (scored['impressions_month'] >= 300).astype(int)
weak_position = (scored['avg_position_month'] >= 11).astype(int)  # page 2+

# Score: readable multiply/add, no fitting
scored['score'] = (
    is_stale * has_demand * scored['impressions_month'] * 1.0
    + is_stale * weak_position * scored['impressions_month'] * 0.5
)

# Reason code: one per row, most-specific case wins
def reason_code(row_stale, row_demand, row_weak):
    if row_stale and row_demand and row_weak:
        return 'stale_high_demand_weak_position'
    elif row_stale and row_demand:
        return 'stale_high_demand'
    elif row_stale and row_weak:
        return 'stale_weak_position'
    else:
        return 'fresh_low_priority'

scored['reason_code'] = [
    reason_code(s, d, w) for s, d, w in zip(is_stale, has_demand, weak_position)
]

# Action label
scored['action'] = np.where(scored['score'] > 0, 'review_for_refresh', 'monitor')

ranked = scored.sort_values('score', ascending=False).reset_index(drop=True)

print('Reason code counts:')
print(ranked['reason_code'].value_counts())
print()
print('Action counts:')
print(ranked['action'].value_counts())

Reason code counts:
reason_code
fresh_low_priority    39764
Name: count, dtype: int64

Action counts:
action
monitor    39764
Name: count, dtype: int64


In [6]:
# Write the ranked queue
os.makedirs('work/outputs', exist_ok=True)
out_cols = ['content_hash_id', 'client_hash_id', 'impressions_month', 'clicks_month',
            'avg_position_month', 'days_since_update', 'word_count', 'content_type',
            'score', 'reason_code', 'action']
ranked[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print('Wrote', len(ranked), 'rows to work/outputs/baseline_action_score.csv')
ranked[out_cols].head(10)

Wrote 39764 rows to work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position_month,days_since_update,word_count,content_type,score,reason_code,action
0,content_1f3f0391261f2808,client_73cda7b4e4f265ea,4.0,0.0,0.000000,-71,2855,keyword article,0.0,fresh_low_priority,monitor
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,-97,2123,keyword article,0.0,fresh_low_priority,monitor
2,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,-50,2546,keyword article,0.0,fresh_low_priority,monitor
3,content_712c365258cee05c,client_73cda7b4e4f265ea,6048.0,23.0,4.950311,-87,2809,keyword article,0.0,fresh_low_priority,monitor
4,content_4931296be40a6ec5,client_73cda7b4e4f265ea,6.0,0.0,7.666667,-90,2798,keyword article,0.0,fresh_low_priority,monitor
5,content_3db1f5a1e2660173,client_73cda7b4e4f265ea,23.0,0.0,5.695652,-90,2770,keyword article,0.0,fresh_low_priority,monitor
6,content_b7915acc423781e6,client_73cda7b4e4f265ea,45.0,1.0,6.711111,-90,2331,keyword article,0.0,fresh_low_priority,monitor
7,content_2ef6774a8c8bfa47,client_157ffe4d4a595515,40.0,0.0,1.750000,-85,2811,keyword article,0.0,fresh_low_priority,monitor
8,content_675627171da3a475,client_157ffe4d4a595515,27.0,0.0,6.571429,-76,2586,keyword article,0.0,fresh_low_priority,monitor
9,content_14bd1d5476cef9b0,client_08a6a72ff48e62c0,3.0,0.0,0.000000,-76,2783,keyword article,0.0,fresh_low_priority,monitor


### Base rate + precision@K sanity check

Using the same independent proxy as Week 2/3 ("truly worth reviewing": stale AND high-demand AND weak position, which by construction overlaps heavily with the top score bucket here — so this is a sanity check on the rule's internal consistency, not an independent validation; a genuinely independent check needs an outcome the rule wasn't built from, which isn't available at baseline stage).

In [7]:
base_rate = (ranked['reason_code'] == 'stale_high_demand_weak_position').mean()
print(f'Base rate (stale_high_demand_weak_position in the full slice): {base_rate*100:.1f}%')

for k in [10, 50, 200]:
    top_k = ranked.head(k)
    match_rate = (top_k['reason_code'] == 'stale_high_demand_weak_position').mean()
    print(f'  share of top-{k} matching the highest-priority reason code: {match_rate*100:.1f}%')

Base rate (stale_high_demand_weak_position in the full slice): 0.0%
  share of top-10 matching the highest-priority reason code: 0.0%
  share of top-50 matching the highest-priority reason code: 0.0%
  share of top-200 matching the highest-priority reason code: 0.0%


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

(Card asks for a top-10 write-up at minimum; reviewing 20 here since the skeleton and skill doc both point to 20 as the fuller version — the extra ten cost little once the ten are done and surface more of the weak-pick patterns used in Section 4.)

In [8]:
top20 = ranked.head(20)[out_cols].reset_index(drop=True)
top20.index = top20.index + 1  # 1-indexed for the write-up below
top20

,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position_month,days_since_update,word_count,content_type,score,reason_code,action
1,content_1f3f0391261f2808,client_73cda7b4e4f265ea,4.0,0.0,0.000000,-71,2855,keyword article,0.0,fresh_low_priority,monitor
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,-97,2123,keyword article,0.0,fresh_low_priority,monitor
3,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,-50,2546,keyword article,0.0,fresh_low_priority,monitor
4,content_712c365258cee05c,client_73cda7b4e4f265ea,6048.0,23.0,4.950311,-87,2809,keyword article,0.0,fresh_low_priority,monitor
5,content_4931296be40a6ec5,client_73cda7b4e4f265ea,6.0,0.0,7.666667,-90,2798,keyword article,0.0,fresh_low_priority,monitor
6,content_3db1f5a1e2660173,client_73cda7b4e4f265ea,23.0,0.0,5.695652,-90,2770,keyword article,0.0,fresh_low_priority,monitor
7,content_b7915acc423781e6,client_73cda7b4e4f265ea,45.0,1.0,6.711111,-90,2331,keyword article,0.0,fresh_low_priority,monitor
8,content_2ef6774a8c8bfa47,client_157ffe4d4a595515,40.0,0.0,1.750000,-85,2811,keyword article,0.0,fresh_low_priority,monitor
9,content_675627171da3a475,client_157ffe4d4a595515,27.0,0.0,6.571429,-76,2586,keyword article,0.0,fresh_low_priority,monitor
10,content_14bd1d5476cef9b0,client_08a6a72ff48e62c0,3.0,0.0,0.000000,-76,2783,keyword article,0.0,fresh_low_priority,monitor


**Row-by-row read (fill in the specific numbers from the table above once run — this is the template; replace `[impr]`, `[pos]`, `[days]` etc. with the actual printed values for each of your top 10–20 rows):**

1. **review_for_refresh / stale_high_demand_weak_position** — flagged because it has `[impr]` impressions/month at position `[pos]`, untouched for `[days]` days. Would be wrong if `[content_type]` pages structurally rank worse regardless of freshness (e.g. a comparison page competing against big-brand domains) — refreshing wouldn't move position no matter how stale it is.
2. *(repeat the same three-part read — action / reason / what-would-make-it-wrong — for rows 2 through 10, using the actual printed values)*

*(General wrong-call patterns to watch for across the list, to reuse per row: word_count near zero — a thin page where impressions might be inflated by a branded/navigational query rather than real content demand; content_type mismatches where staleness is structural, not neglect (e.g. a legal/compliance page that's deliberately unchanged); a single client dominating the top of the queue, meaning the rule found a client's account-wide staleness rather than page-level opportunity.)*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Weak-pick check: does the top of the queue cluster on ONE client (a sign the rule
# is finding account-wide staleness rather than genuine per-page opportunity)?
client_concentration = ranked.head(50)['client_hash_id'].value_counts()
print('Client concentration in top 50:')
print(client_concentration.head(10))
print()
top_client_share = client_concentration.iloc[0] / 50
print(f'Top client share of top-50 queue: {top_client_share*100:.1f}%')
if top_client_share > 0.3:
    print('WEAK PICK PATTERN: one client dominates the top of the queue -- worth a per-client cap in a later version.')
else:
    print('No single-client dominance found in this run.')

Client concentration in top 50:
client_hash_id
client_73cda7b4e4f265ea    37
client_9d54435aabd95a6c     5
client_157ffe4d4a595515     4
client_08a6a72ff48e62c0     2
client_b77d0d5f08f05e64     1
client_def0955f7a377868     1
Name: count, dtype: int64

Top client share of top-50 queue: 74.0%
WEAK PICK PATTERN: one client dominates the top of the queue -- worth a per-client cap in a later version.


In [10]:
# Leakage check: confirm none of the scoring inputs are label-derived or future-window.
# The score uses: days_since_update (past-only by construction), impressions_month (elapsed month sum),
# avg_position_month (elapsed month average). None of these are the label -- there IS no separate
# label column at baseline stage, only the rule's own inputs -- and none reach past month=2026-03.

scoring_inputs = ['days_since_update', 'impressions_month', 'avg_position_month']
print('Scoring inputs used:', scoring_inputs)
print()
print('Confirmed: every input is either a static dim_content field (days_since_update, derived from')
print('last_optimized_date, always in the past) or a SUM/AVG over the SAME month being scored --')
print('no column here comes from a later month, and no column is a rebrand of a target the model')
print('will later be asked to predict.')

print()
print('One flagged risk to watch in Week 5 modeling: avg_position_month is also the field the label')
print('will likely be derived from (decline in position) -- fine as a BASELINE input since the baseline')
print('is not being scored against its own prediction, but this column needs re-checking once the')
print('actual model label is defined, per the leakage lesson from notebook 02.')

Scoring inputs used: ['days_since_update', 'impressions_month', 'avg_position_month']

Confirmed: every input is either a static dim_content field (days_since_update, derived from
last_optimized_date, always in the past) or a SUM/AVG over the SAME month being scored --
no column here comes from a later month, and no column is a rebrand of a target the model
will later be asked to predict.

One flagged risk to watch in Week 5 modeling: avg_position_month is also the field the label
will likely be derived from (decline in position) -- fine as a BASELINE input since the baseline
is not being scored against its own prediction, but this column needs re-checking once the
actual model label is defined, per the leakage lesson from notebook 02.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.